# Business Entity Resolution: v3 (v1 + multilingual language model re-check)

**Settings (right-hand panel → Session options):** Accelerator **GPU T4 x2**, Internet **On** (needed once to download the language model).
**How to run:** click **Save Version** (top right), choose **Save & Run All (Commit)**, then **Save**. It runs on Kaggle's servers even if you close the tab.

What is new compared with v1:
1. Shortlist and v1's similarity model are the same as v1 (trained on the same 150k businesses minus the same 30k held-out ones).
2. For each business, its best ~12 candidates are re-checked by a **multilingual language model** (MiniLM, 118M parameters, Apache-2.0).
   It reads both records as text, side by side, and is fine-tuned on the training pairs. It can read Hindi, Tamil, Telugu, Kannada, Gujarati and French directly and copes with misspellings.
3. A small final model combines v1's features, v1's chance and the language model's score, and a cut-off tuned on held-out businesses decides.

Cells: 1 install, 2 v1 pipeline code, 3 settings, 4 v3 code, 5 **train** (prints `BEST val F0.5`; v1 was 0.9529), 6 **predict** test and write `output/matching_results.tsv`, 7 official format check.

Expect about 2 hours for training and 3 to 4 hours for prediction (well inside Kaggle's 12-hour limit). No external data or APIs are used, only the competition files and the open language model.

In [ ]:
!pip install -q rapidfuzz sparse_dot_topn unidecode polars lightgbm
!nvidia-smi -L

In [ ]:
# =====================================================================
# Business Entity Resolution: full pipeline (clean -> shortlist -> features -> LightGBM -> decide)
# No external data or services are used anywhere. Only the provided files.
# =====================================================================
import os, re, gc, glob, time, json
import numpy as np
import pandas as pd
import polars as pl
from multiprocessing import Pool
from unidecode import unidecode
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.process import cpdist
import lightgbm as lgb

N_CPU = os.cpu_count() or 4

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

# ---------------------------------------------------------------------
# 1. Loading. Files are tab-separated, all columns read as text.
#    Handles files split into *_part_N.tsv (BOM / CRLF / repeated header).
# ---------------------------------------------------------------------
def find_file(root, name):
    hits = sorted(glob.glob(f"{root}/**/{name}.tsv", recursive=True))
    if hits:
        return [hits[0]]
    parts = glob.glob(f"{root}/**/{name}_part_*.tsv", recursive=True)
    parts.sort(key=lambda p: int(re.findall(r"_part_(\d+)", p)[-1]))
    if not parts:
        raise FileNotFoundError(f"{name}.tsv not found under {root}")
    return parts

def load(root, name):
    frames, header = [], None
    for p in find_file(root, name):
        txt = open(p, encoding="utf-8-sig").read().replace("\r\n", "\n").replace("\r", "")
        if txt.startswith(("entity_id\t", "source1_entity_id\t")):
            header, txt = txt.split("\n", 1)
        frames.append(txt)
    body = "".join(f if f.endswith("\n") else f + "\n" for f in frames)
    df = pl.read_csv((header + "\n" + body).encode(), separator="\t", quote_char=None, infer_schema_length=0)
    return df.fill_null("").to_pandas()

# ---------------------------------------------------------------------
# 2. Cleaning (normalisation). Hand-written rules only.
# ---------------------------------------------------------------------
ABBR = {"rd":"road","st":"street","ave":"avenue","av":"avenue","blvd":"boulevard","bd":"boulevard","hwy":"highway",
        "dr":"drive","ln":"lane","ct":"court","pl":"place","pkwy":"parkway","cir":"circle","ste":"suite","fl":"floor",
        "apt":"apartment","no":"number","nr":"near","opp":"opposite","bldg":"building","ind":"industrial",
        "dist":"district","r":"rue","ch":"chemin","n":"north","s":"south","e":"east","w":"west","mt":"mount",
        "ft":"fort","hts":"heights","sq":"square","imp":"impasse","all":"allee"}
NULLS = {"null", "na", "none", "nan"}
# legal words, written in "skeleton" form (see skel_tok): covers Limited/Ltd/limittedd, Private/Pvt/praaivett, LLC, SARL ...
LEGAL_SK = {"lmtd","ltd","prwt","pwt","llc","llp","inc","ncrprtd","crp","crprtn","c","cmpny","plc","srl","ss","sa",
            "sc","erl","snc","gmbh","t","m","s","d","dba","lp","pc","pllc","pa","ms","and"}

def base(s):
    s = unidecode(str(s)).lower()                                   # also turns Hindi/Telugu/... script into Latin letters
    s = re.sub(r"<?null>?|n/a", " ", s)
    s = re.sub(r"https?://|www\.|\.(com|net|org|in|co|io|biz|fr|us)\b", " ", s)
    s = s.replace("&", " and ").replace("+", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def skel_tok(t):
    """Spelling-robust 'skeleton' of a word: drop vowels, merge ph/f, bh/b ..., collapse double letters.
    'limited' and the transliterated 'limittedd' both become 'lmtd'."""
    if not t or t.isdigit():
        return t
    t = t.replace("0", "o").replace("1", "l") if re.search(r"[a-z]", t) else t   # leetspeak: C1ean -> Clean
    t = re.sub(r"ph", "f", t)
    t = re.sub(r"([bkgtdcs])h", r"\1", t)
    t = t.replace("v", "w").replace("q", "k").replace("z", "s").replace("x", "ks")
    t = t[0] + re.sub(r"[aeiouy]", "", t[1:])
    return re.sub(r"(.)\1+", r"\1", t)

US = {"alabama":"al","alaska":"ak","arizona":"az","arkansas":"ar","california":"ca","colorado":"co","connecticut":"ct",
"delaware":"de","florida":"fl","georgia":"ga","hawaii":"hi","idaho":"id","illinois":"il","indiana":"in","iowa":"ia",
"kansas":"ks","kentucky":"ky","louisiana":"la","maine":"me","maryland":"md","massachusetts":"ma","michigan":"mi",
"minnesota":"mn","mississippi":"ms","missouri":"mo","montana":"mt","nebraska":"ne","nevada":"nv","new hampshire":"nh",
"new jersey":"nj","new mexico":"nm","new york":"ny","north carolina":"nc","north dakota":"nd","ohio":"oh","oklahoma":"ok",
"oregon":"or","pennsylvania":"pa","rhode island":"ri","south carolina":"sc","south dakota":"sd","tennessee":"tn",
"texas":"tx","utah":"ut","vermont":"vt","virginia":"va","washington":"wa","west virginia":"wv","wisconsin":"wi",
"wyoming":"wy","district of columbia":"dc","puerto rico":"pr"}
IN = {"andhra pradesh":"ap","arunachal pradesh":"ar","assam":"as","bihar":"br","chhattisgarh":"cg","goa":"ga",
"gujarat":"gj","haryana":"hr","himachal pradesh":"hp","jharkhand":"jh","karnataka":"ka","kerala":"kl",
"madhya pradesh":"mp","maharashtra":"mh","manipur":"mn","meghalaya":"ml","mizoram":"mz","nagaland":"nl","odisha":"od",
"orissa":"od","punjab":"pb","rajasthan":"rj","sikkim":"sk","tamil nadu":"tn","telangana":"ts","tripura":"tr",
"uttar pradesh":"up","uttarakhand":"uk","uttaranchal":"uk","west bengal":"wb","delhi":"dl","new delhi":"dl",
"jammu and kashmir":"jk","ladakh":"la","puducherry":"py","pondicherry":"py","chandigarh":"ch",
"andaman and nicobar islands":"an","lakshadweep":"ld"}
MERGE_ST = {"ts": "ap"}            # Telangana was split from Andhra Pradesh; the data mixes them
_STATE = {}
for _d in (US, IN):
    for _full, _code in _d.items():
        _STATE[tuple(skel_tok(t) for t in _full.split())] = _code
_CODES = set(US.values()) | set(IN.values())

def states_of(raw_addr):
    """All state/province codes mentioned in an address ('' when none). Country-agnostic: unknown countries get ''."""
    found = set()
    for seg in unidecode(str(raw_addr)).lower().split(","):
        toks = re.sub(r"[^a-z ]", " ", seg).split()
        if not toks:
            continue
        if len(toks) == 1 and toks[0] in _CODES:
            found.add(toks[0]); continue
        sk = [skel_tok(t) for t in toks]
        for n in (4, 3, 2, 1):
            for i in range(len(sk) - n + 1):
                key = tuple(sk[i:i + n])
                if key in _STATE and (n > 1 or len(key[0]) >= 3):
                    found.add(_STATE[key])
    return "|".join(sorted({MERGE_ST.get(x, x) for x in found}))

def _clean_rows(rows):
    out = []
    for name, addr in rows:
        nb = base(name)
        sk = [skel_tok(t) for t in nb.split()]
        core = [t for t in sk if t not in LEGAL_SK] or sk
        an = " ".join(ABBR.get(t, t) for t in base(addr).split() if t not in NULLS)
        out.append((nb, " ".join(core), "".join(core), an,
                    " ".join(skel_tok(t) for t in an.split()), states_of(addr)))
    return out

def prepare(df):
    """Adds the cleaned columns. Runs on all CPU cores."""
    rows = list(zip(df.business_name.tolist(), df.business_address.tolist()))
    step = 50_000
    with Pool(N_CPU) as pool:
        res = pool.map(_clean_rows, [rows[i:i + step] for i in range(0, len(rows), step)])
    res = [r for chunk in res for r in chunk]
    cols = ["name_b", "name_sk", "name_skc", "addr_n", "addr_sk", "states"]
    for j, c in enumerate(cols):
        df[c] = [r[j] for r in res]
    df.drop(columns=["business_address"], inplace=True)
    df["nonascii"] = df.business_name.str.contains(r"[^\x00-\x7f]", regex=True).astype(np.int8)
    df.drop(columns=["business_name"], inplace=True)
    return df.reset_index(drop=True)

# ---------------------------------------------------------------------
# 3. Shortlist ("blocking"). For each Source 1 record, look only inside the same country and
#    the same state, and keep the top-k most similar names and the top-k most similar addresses.
#    Records whose state could not be read form a separate pool per country.
# ---------------------------------------------------------------------
def _vec_name():
    return TfidfVectorizer(analyzer="char", ngram_range=(3, 3), dtype=np.float32, sublinear_tf=True)
def _vec_addr():
    return TfidfVectorizer(analyzer="word", token_pattern=r"\S+", ngram_range=(1, 2), dtype=np.float32, sublinear_tf=True)

class Index:
    """TF-IDF indexes over one pool of candidate records (one country of Source 2 or Source 3)."""
    def __init__(self, B, k_state=15, k_unk=10):
        self.B, self.k_state, self.k_unk = B, k_state, k_unk
        self.cache = {}
        st = B.states.str.split("|")
        self.members = {}
        for pos, lst in zip(range(len(B)), st):
            for s in lst:
                if s:
                    self.members.setdefault(s, []).append(pos)
        self.unknown = np.flatnonzero(B.states.to_numpy() == "")

    def _build(self, key, pos):
        if key in self.cache:
            return self.cache[key]
        pos = np.asarray(pos, dtype=np.int64)
        entry = {"pos": pos}
        for col, mk in (("name_skc", _vec_name), ("addr_sk", _vec_addr)):
            try:
                v = mk(); M = v.fit_transform(self.B[col].to_numpy()[pos])
                entry[col] = (v, M.T.tocsr())
            except ValueError:            # empty vocabulary, e.g. every address blank
                entry[col] = None
        self.cache[key] = entry
        return entry

    def query(self, A, thr=0.1, chunk=40_000):
        """Returns DataFrame(a, b, sim_name, sim_addr); a = row in A, b = row in self.B."""
        out = []
        ast = A.states.str.split("|").tolist()
        groups = {}
        for i, lst in enumerate(ast):
            for s in lst:
                groups.setdefault(s, []).append(i)
        jobs = []
        for s, rows in groups.items():
            if s == "":
                jobs.append(("__all__", np.arange(len(self.B)), rows, self.k_state))
            elif s in self.members:
                jobs.append((s, self.members[s], rows, self.k_state))
        known = [i for i, lst in enumerate(ast) if lst != [""]]
        if len(self.unknown) and known:
            jobs.append(("__unknown__", self.unknown, known, self.k_unk))
        for key, pos, rows, k in jobs:
            e = self._build(key, pos)
            rows = np.asarray(rows)
            for col, tag in (("name_skc", 0), ("addr_sk", 1)):
                if e[col] is None:
                    continue
                v, BT = e[col]
                texts = A[col].to_numpy()[rows]
                for i in range(0, len(rows), chunk):
                    C = sp_matmul_topn(v.transform(texts[i:i + chunk]), BT, top_n=k, threshold=thr,
                                       n_threads=N_CPU).tocoo()
                    out.append((rows[i + C.row], e["pos"][C.col], C.data, tag))
        if not out:
            return pd.DataFrame({"a": [], "b": [], "sim_name": [], "sim_addr": []})
        a = np.concatenate([o[0] for o in out]).astype(np.int64)
        b = np.concatenate([o[1] for o in out]).astype(np.int64)
        sim = np.concatenate([o[2] for o in out]).astype(np.float32)
        tag = np.concatenate([np.full(len(o[0]), o[3], np.int8) for o in out])
        df = pd.DataFrame({"key": a * len(self.B) + b,
                           "sim_name": np.where(tag == 0, sim, np.nan),
                           "sim_addr": np.where(tag == 1, sim, np.nan)})
        df = df.groupby("key", sort=False).max().reset_index()
        df["a"] = (df.key // len(self.B)).astype(np.int32)
        df["b"] = (df.key % len(self.B)).astype(np.int32)
        return df.drop(columns="key")

# ---------------------------------------------------------------------
# 4. Features: how similar is each (Source 1, candidate) pair?
# ---------------------------------------------------------------------
FULL = [("tset", fuzz.token_set_ratio), ("tsort", fuzz.token_sort_ratio), ("ratio", fuzz.ratio),
        ("partial", fuzz.partial_ratio), ("jw", JaroWinkler.normalized_similarity)]

def _nums(s):
    return set(re.findall(r"\d+", s))

def features(P, A, B, is_s3):
    a, b = P.a.to_numpy(), P.b.to_numpy()
    g = lambda D, col, idx: D[col].to_numpy()[idx]
    f = pd.DataFrame({"sim_name": P.sim_name.to_numpy(), "sim_addr": P.sim_addr.to_numpy()})
    for col, pre, scorers in (("name_b", "nb", FULL), ("name_sk", "nsk", FULL),
                              ("name_skc", "nskc", [("ratio", fuzz.ratio), ("partial", fuzz.partial_ratio)]),
                              ("addr_sk", "ask", [("tset", fuzz.token_set_ratio), ("ratio", fuzz.ratio),
                                                  ("partial", fuzz.partial_ratio)])):
        x, y = g(A, col, a).tolist(), g(B, col, b).tolist()
        for nm, sc in scorers:
            f[f"{pre}_{nm}"] = cpdist(x, y, scorer=sc, workers=N_CPU).astype(np.float32)
    xa, ya = g(A, "addr_n", a), g(B, "addr_n", b)
    nl = [_nums(s) for s in xa]; nr = [_nums(s) for s in ya]
    f["num_jacc"] = np.array([len(p & q) / len(p | q) if (p | q) else -1 for p, q in zip(nl, nr)], np.float32)
    f["num_contain"] = np.array([len(p & q) / len(q) if q else -1 for p, q in zip(nl, nr)], np.float32)
    sa, sb = g(A, "states", a), g(B, "states", b)
    f["state_rel"] = np.array([-1 if (not p or not q) else (1 if set(p.split("|")) & set(q.split("|")) else 0)
                               for p, q in zip(sa, sb)], np.int8)
    f["name_exact"] = (g(A, "name_skc", a) == g(B, "name_skc", b)).astype(np.int8)
    f["cand_addr_len"] = np.fromiter((len(s) for s in ya), np.int32, len(ya))
    f["s1_addr_len"] = np.fromiter((len(s) for s in xa), np.int32, len(xa))
    f["cand_name_len"] = np.fromiter((len(s) for s in g(B, "name_b", b)), np.int32, len(b))
    f["s1_name_len"] = np.fromiter((len(s) for s in g(A, "name_b", a)), np.int32, len(a))
    f["cand_nonascii"] = g(B, "nonascii", b)
    f["is_s3"] = np.int8(is_s3)
    # context: how this candidate compares with the other candidates of the same Source 1 record
    v = f.nsk_tset.to_numpy() + f.ask_tset.to_numpy()
    grp = pd.DataFrame({"a": a, "v": v}).groupby("a")["v"]
    f["rank_s1"] = grp.rank(ascending=False, method="min").to_numpy(np.float32)
    f["gap_s1"] = (grp.transform("max").to_numpy() - v).astype(np.float32)
    f["n_cand_s1"] = grp.transform("size").to_numpy(np.int32)
    return f

# ---------------------------------------------------------------------
# 5. Scoring (same formula as the leaderboard) and the final decision rule
# ---------------------------------------------------------------------
def macro_f05(pred, truth, ids):
    sc = []
    for s in ids:
        p, t = pred.get(s, set()), truth.get(s, set())
        if not t:
            sc.append(1.0 if not p else 0.0); continue
        tp = len(p & t)
        if tp == 0:
            sc.append(0.0); continue
        P, R = tp / len(p), tp / len(t)
        sc.append(1.25 * P * R / (0.25 * P + R))
    return float(np.mean(sc))

def decide(pairs, thr):
    """pairs: DataFrame(s1_id, cand_id, p). Keep p >= thr; each S2/S3 record goes to at most one S1 (its best)."""
    d = pairs[pairs.p >= thr].sort_values("p", ascending=False).drop_duplicates("cand_id")
    return d.groupby("s1_id")["cand_id"].apply(set).to_dict()

def write_tsv(path, s1_ids, mapping, col):
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(f"source1_entity_id\t{col}\n")
        for s in s1_ids:
            fh.write(s + "\t" + ",".join(sorted(mapping.get(s, ()))) + "\n")


In [ ]:
ROOT = os.environ.get("ROOT", "/kaggle/input")      # the notebook searches here for the .tsv files
WORK = os.environ.get("WORK", "/kaggle/working")    # models + outputs are saved here
SMOKE = 0             # 1 = quick test on a small slice (for checking the code only)
N_TRAIN_S1 = 150_000 if not SMOKE else 6_000        # training businesses; same sample (seed 0) and same 30k held-out as v1
BATCH = int(os.environ.get("BATCH", 100_000))       # test businesses processed at once (lower it if memory runs out)

# ---- the language model ("cross-encoder") ----
# Multilingual MiniLM: 118M parameters, Apache-2.0 licence, knows 50+ languages incl. Hindi, Tamil, Telugu, French.
CE_MODEL = os.environ.get("CE_MODEL", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
CE_MINP = 0.01          # only candidates whose v1 chance is at least 1% ...
CE_TOPK = 12            # ... and among the business's 12 best (both sources together) are re-checked by the model
MAX_LEN = 96            # tokens per pair (name + address of both records)
CE_EPOCHS = 2           # passes over the training pairs
CE_TRAIN_MAX_PAIRS = 900_000
CE_TRAIN_MINUTES = 80   # safety limit: training stops after this many minutes even if not finished
CE_LR = 5e-5
CE_BS_TRAIN = 64
CE_BS_INFER = 1024


In [ ]:
# =====================================================================
# Version 3: v1's shortlist + v1's model, then a multilingual language model re-checks the best candidates.
# The language model ("cross-encoder") reads BOTH records as text, side by side, and learns from the
# training pairs whether they are the same business. It can read Hindi/Tamil/Telugu/Kannada/Gujarati and
# French directly, and is robust to spelling noise, which hand-written similarity scores are not.
# A small final LightGBM ("blender") combines v1's features, v1's chance and the language model's score.
# =====================================================================
import math, zlib, torch
from sklearn.model_selection import GroupKFold

DEV = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = torch.cuda.device_count()

def tenth(ids):
    """True for a fixed ~10% of business ids (used to hold out rows for early stopping)."""
    return np.array([zlib.crc32(s.encode()) % 10 == 0 for s in ids])

def load_src(name):
    """Loads and cleans one Source 2/3 file once, keeps the raw text for the language model, caches it on disk."""
    path = f"{WORK}/clean_{name}.parquet"
    if os.path.exists(path):
        return pd.read_parquet(path)
    df = load(ROOT, name)
    df["rn"], df["ra"] = df.business_name, df.business_address
    df = prepare(df)
    df.to_parquet(path)
    return df

# ---------- text shown to the language model ----------
_JUNK = re.compile(r"<?\bnull\b>?|\bn/a\b|<NULL>", re.I)
def rec_text(name, addr):
    """'name (latin spelling) | address'. Non-Latin names are given in both scripts."""
    out = []
    for n, a in zip(name, addr):
        n = str(n).strip(); a = re.sub(r"\s+", " ", _JUNK.sub(" ", str(a))).strip(" ,")
        if re.search(r"[^\x00-\x7f]", n):
            n = f"{n} ({unidecode(n).strip()})"
        if re.search(r"[^\x00-\x7f]", a):
            a = unidecode(a)
        out.append(f"{n} | {a}" if a else n)
    return out

def text_lookup(src, ids):
    """Raw text of the given Source 2/3 ids (reads only the needed rows from the cached file)."""
    t = pl.read_parquet(f"{WORK}/clean_{src}.parquet", columns=["entity_id", "rn", "ra"])
    t = pl.DataFrame({"entity_id": pd.unique(np.asarray(ids))}).join(t, on="entity_id", how="left").to_pandas()
    return pd.Series(rec_text(t.rn.fillna(""), t.ra.fillna("")), index=t.entity_id)

# ---------- per-business context features (computed the same way in training, validation and test) ----------
def add_ctx(D, key):
    g = D.groupby(key)
    D["p1_rank"] = g.p1.rank(ascending=False, method="min").astype(np.float32)
    D["p1_gap"] = (g.p1.transform("max") - D.p1).astype(np.float32)
    D["n_sel"] = g.p1.transform("size").astype(np.float32)
    if "ce" in D:
        D["ce_npos"] = (D.ce > 0).astype(np.float32)
        g = D.groupby(key)
        D["ce_npos"] = g.ce_npos.transform("sum").astype(np.float32)     # how many candidates the model likes
        D["ce_rank"] = g.ce.rank(ascending=False, method="min").astype(np.float32)
        D["ce_max"] = g.ce.transform("max").astype(np.float32)
        D["ce_gap"] = (D.ce_max - D.ce).astype(np.float32)
    return D

def select_for_ce(D, key):
    """Candidates the language model will re-check: v1 chance >= CE_MINP and among the business's CE_TOPK best."""
    r = D.groupby(key).p1.rank(ascending=False, method="first")
    return ((D.p1 >= CE_MINP) & (r <= CE_TOPK)).to_numpy()

# ---------- the language model ----------
def ce_load(name):
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=1)
    return tok, model.to(DEV)

def _encode(tok, ta, tb):
    enc = tok(list(ta), list(tb), truncation=True, max_length=MAX_LEN)
    return enc["input_ids"]

def _batch(tok, seqs):
    L = max(len(s) for s in seqs)
    ids = np.full((len(seqs), L), tok.pad_token_id, np.int64); att = np.zeros((len(seqs), L), np.int64)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = s; att[i, :len(s)] = 1
    return {"input_ids": torch.from_numpy(ids).to(DEV), "attention_mask": torch.from_numpy(att).to(DEV)}

def _amp():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda")

def ce_train(tok, model, ta, tb, y):
    from transformers import get_linear_schedule_with_warmup
    seqs = _encode(tok, ta, tb); y = np.asarray(y, np.float32)
    log("language model: training on", len(y), "pairs,", f"{y.mean():.1%} true matches,", N_GPU, "GPU(s)")
    net = torch.nn.DataParallel(model) if N_GPU > 1 else model
    steps_ep = math.ceil(len(y) / CE_BS_TRAIN); steps = CE_EPOCHS * steps_ep
    opt = torch.optim.AdamW(model.parameters(), lr=CE_LR, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    lossf = torch.nn.BCEWithLogitsLoss(); t0 = time.time(); step = 0; run = 0.0
    net.train()
    for ep in range(CE_EPOCHS):
        perm = np.random.default_rng(ep).permutation(len(y))
        for i in range(0, len(y), CE_BS_TRAIN):
            j = perm[i:i + CE_BS_TRAIN]
            b = _batch(tok, [seqs[k] for k in j]); yb = torch.from_numpy(y[j]).to(DEV)
            with _amp():
                out = net(**b).logits.squeeze(-1)
            loss = lossf(out.float(), yb)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); step += 1
            run = 0.98 * run + 0.02 * loss.item() if step > 1 else loss.item()
            if step % 500 == 0 or step == steps:
                log(f"  epoch {ep + 1}/{CE_EPOCHS} step {step}/{steps} loss {run:.4f} ({(time.time() - t0) / 60:.1f} min)")
            if time.time() - t0 > CE_TRAIN_MINUTES * 60:
                log("  time limit reached, stopping training at step", step); break
        else:
            continue
        break
    model.eval()
    return model

@torch.inference_mode()
def ce_score(tok, model, ta, tb):
    """Language-model score (a 'logit': >0 means it thinks it is a match) for each pair."""
    net = torch.nn.DataParallel(model) if N_GPU > 1 else model
    net.eval()
    seqs = _encode(tok, ta, tb)
    order = np.argsort([len(s) for s in seqs], kind="stable")      # similar lengths together = less padding = faster
    out = np.zeros(len(seqs), np.float32)
    for i in range(0, len(order), CE_BS_INFER):
        j = order[i:i + CE_BS_INFER]
        with _amp():
            out[j] = net(**_batch(tok, [seqs[k] for k in j])).logits.squeeze(-1).float().cpu().numpy()
    return out

# ---------- final model that combines everything ----------
BLEND_PARAMS = dict(learning_rate=0.03, num_leaves=63, min_child_samples=50, subsample=0.8, subsample_freq=1,
                    colsample_bytree=0.8, n_jobs=N_CPU, verbose=-1)

def p1_of(boosters, F):
    x = F.to_numpy(np.float32)
    return np.mean([b.predict(x, num_threads=N_CPU) for b in boosters], axis=0).astype(np.float32)


In [ ]:
# ---------- 1. shortlist + v1 features for the same 150k training businesses as v1 ----------
for src in ("train_source2", "train_source3"):          # clean both sources first (saved to disk)
    log("cleaning", src); B = load_src(src); log("cleaned", src, B.shape); del B; gc.collect()
log("loading train Source 1 + ground truth")
s1 = load(ROOT, "train_source1"); gt = load(ROOT, "train_ground_truth")
s1 = s1.sample(N_TRAIN_S1, random_state=0).reset_index(drop=True)
truth = {r.source1_entity_id: set(x for x in r.matched_entity_ids.split(",") if x) for r in gt.itertuples()}
del gt; gc.collect()
s1_text = pd.Series(rec_text(s1.business_name, s1.business_address), index=s1.entity_id.to_numpy())
A = prepare(s1); log("prepared S1 sample", A.shape)
rng = np.random.default_rng(42)                          # identical held-out businesses to v1 (0.9529)
val_ids = set(rng.choice(A.entity_id.to_numpy(), int(0.2 * len(A)), replace=False))
linked = set().union(*(truth[s] for s in A.entity_id))
feats, metas = [], []
for src, is_s3 in (("train_source2", 0), ("train_source3", 1)):
    B = load_src(src).drop(columns=["rn", "ra"])
    if SMOKE:                                            # quick test: a quarter of the pool plus the true matches
        B = B[B.entity_id.isin(linked) | (np.random.default_rng(1).random(len(B)) < 0.25)].reset_index(drop=True)
    for country in A.country.unique():
        Ac = A[A.country == country].reset_index(drop=True); Bc = B[B.country == country].reset_index(drop=True)
        if len(Ac) == 0 or len(Bc) == 0: continue
        P = Index(Bc).query(Ac); log(src, country, "pairs", len(P))
        F = features(P, Ac, Bc, is_s3)
        m = pd.DataFrame({"s1_id": Ac.entity_id.to_numpy()[P.a], "cand_id": Bc.entity_id.to_numpy()[P.b], "src": src})
        m["label"] = [c in truth.get(s, ()) for s, c in zip(m.s1_id, m.cand_id)]
        feats.append(F); metas.append(m); gc.collect()
    del B; gc.collect()
X = pd.concat(feats, ignore_index=True); M = pd.concat(metas, ignore_index=True); del feats, metas; gc.collect()
y = M.label.to_numpy().astype(int)
all_true = sum(len(truth[s]) for s in A.entity_id)
log("pairs", len(M), "per S1", round(len(M) / len(A), 1), "| shortlist recall", round(y.sum() / all_true, 4))

# ---------- 2. v1's model, trained on two halves so every training pair gets an honest (unseen) score ----------
iv = M.s1_id.isin(val_ids).to_numpy()
tr_ids = np.array(sorted(set(A.entity_id) - val_ids))
half = dict(zip(tr_ids, np.random.default_rng(7).integers(0, 2, len(tr_ids))))
fold = M.s1_id.map(half).fillna(-1).to_numpy().astype(int)
V1_PARAMS = dict(n_estimators=4000, learning_rate=0.05, num_leaves=255, min_child_samples=100, subsample=0.8,
                 subsample_freq=1, colsample_bytree=0.8, n_jobs=N_CPU, verbose=-1)
p1 = np.zeros(len(M), np.float32); boosters = []
for k in (0, 1):
    fit = fold == 1 - k
    stop = fit & tenth(M.s1_id)      # 10% of the half, for early stopping
    mdl = lgb.LGBMClassifier(**V1_PARAMS)
    mdl.fit(X[fit & ~stop], y[fit & ~stop], eval_set=[(X[stop], y[stop])],
            callbacks=[lgb.early_stopping(100, verbose=False)])
    boosters.append(mdl.booster_); mdl.booster_.save_model(f"{WORK}/stage1_{k}.txt")
    p1[fold == k] = mdl.predict_proba(X[fold == k])[:, 1]
    log(f"v1 model on half {k + 1}/2: {mdl.best_iteration_} trees")
p1[iv] = p1_of(boosters, X[iv]); M["p1"] = p1
vl = sorted(val_ids); vt = {s: truth[s] for s in vl}
log("baseline all-empty", round(macro_f05({}, vt, vl), 4))
log("ceiling (perfect model on this shortlist)",
    round(macro_f05(M[iv & (y == 1)].groupby("s1_id").cand_id.apply(set).to_dict(), vt, vl), 4))
pv = M[iv][["s1_id", "cand_id"]].assign(p=M.p1[iv])
res1 = [(round(t, 2), macro_f05(decide(pv, t), vt, vl)) for t in np.arange(0.40, 0.86, 0.05)]
log("v1-style model alone: BEST threshold %s val F0.5 %.4f" % max(res1, key=lambda r: r[1]))

# ---------- 3. candidates the language model re-checks ----------
sel = select_for_ce(M, "s1_id")
S = M[sel].reset_index(drop=True); XS = X[sel].reset_index(drop=True)
del X; gc.collect()
log("re-check set:", len(S), "pairs,", round(len(S) / len(A), 1), "per business | true matches kept",
    round(S.label.sum() / all_true, 4), "(shortlist had", round(y.sum() / all_true, 4), ")")
ta = s1_text.reindex(S.s1_id).to_numpy(); tb = np.empty(len(S), object)
for src in ("train_source2", "train_source3"):
    m = (S.src == src).to_numpy(); tb[m] = text_lookup(src, S.cand_id[m]).reindex(S.cand_id[m]).to_numpy()
for i in np.flatnonzero(S.label.to_numpy())[:3].tolist() + np.flatnonzero(~S.label.to_numpy())[:2].tolist():
    print("   example:", ta[i], "  <->  ", tb[i], "  match" if S.label[i] else "  not a match")

# ---------- 4. fine-tune the language model on training businesses (never on the held-out ones) ----------
trm = ~S.s1_id.isin(val_ids).to_numpy()
tr_rows = np.flatnonzero(trm)
if len(tr_rows) > CE_TRAIN_MAX_PAIRS:                    # keep whole businesses when sampling
    ids = S.s1_id.to_numpy()[tr_rows]; u = pd.unique(ids)
    keep = set(np.random.default_rng(3).choice(u, int(len(u) * CE_TRAIN_MAX_PAIRS / len(tr_rows)), replace=False))
    tr_rows = tr_rows[pd.Series(ids).isin(keep).to_numpy()]
log("loading language model", CE_MODEL)
tok, cem = ce_load(CE_MODEL)
cem = ce_train(tok, cem, ta[tr_rows], tb[tr_rows], S.label.to_numpy()[tr_rows])
cem.save_pretrained(f"{WORK}/ce_model"); tok.save_pretrained(f"{WORK}/ce_model")
vrows = np.flatnonzero(~trm)
S["ce"] = np.nan
t0 = time.time(); S.loc[vrows, "ce"] = ce_score(tok, cem, ta[vrows], tb[vrows])
log("scored", len(vrows), "held-out pairs in", round(time.time() - t0), "s")

# ---------- 5. final model (blender), cross-validated on the held-out businesses ----------
D = pd.concat([XS.iloc[vrows].reset_index(drop=True),
               S.iloc[vrows][["s1_id", "cand_id", "label", "p1", "ce"]].reset_index(drop=True)], axis=1)
D = add_ctx(D, "s1_id")
BLEND = list(XS.columns) + ["p1", "ce", "p1_rank", "p1_gap", "n_sel", "ce_npos", "ce_rank", "ce_max", "ce_gap"]
yd = D.label.to_numpy().astype(int); oof = np.zeros(len(D)); iters = []
for f, (a, b) in enumerate(GroupKFold(5).split(D, yd, D.s1_id)):
    stop = tenth(D.s1_id.iloc[a])
    mdl = lgb.LGBMClassifier(n_estimators=3000, **BLEND_PARAMS)
    mdl.fit(D[BLEND].iloc[a[~stop]], yd[a[~stop]], eval_set=[(D[BLEND].iloc[a[stop]], yd[a[stop]])],
            callbacks=[lgb.early_stopping(100, verbose=False)])
    oof[b] = mdl.predict_proba(D[BLEND].iloc[b])[:, 1]; iters.append(mdl.best_iteration_)
log("blender trees per fold", iters)
grid = np.arange(0.30, 0.91, 0.025)
def best_of(p):
    r = [(round(t, 3), macro_f05(decide(D[["s1_id", "cand_id"]].assign(p=p), t), vt, vl)) for t in grid]
    return max(r, key=lambda z: z[1]), r
(b1, _), (b2, _) = best_of(D.p1.to_numpy()), best_of(1 / (1 + np.exp(-D.ce.to_numpy())))
log("v1 model on re-check set only: threshold %s val F0.5 %.4f" % b1)
log("language model alone:          threshold %s val F0.5 %.4f" % b2)
(best_t, best_s), res = best_of(oof)
for t, s in res[::2]: log("blend threshold", t, "val F0.5", round(s, 4))
log(f"BEST val F0.5 {best_s:.4f} at threshold {best_t} (v1 was 0.9529 on the same held-out businesses)")
final = lgb.LGBMClassifier(n_estimators=max(50, int(np.mean(iters))), **BLEND_PARAMS).fit(D[BLEND], yd)
final.booster_.save_model(f"{WORK}/blender.txt")
use_blend = best_s > b1[1]                                # safety net: never worse than v1 on the same pairs
if not use_blend: log("language model did not help here, the test step will use the v1 model's chance instead")
json.dump({"use_blend": bool(use_blend), "threshold": best_t if use_blend else b1[0], "val_f05": max(best_s, b1[1]), "v1_features": list(XS.columns), "blend_features": BLEND},
          open(f"{WORK}/model_v3_meta.json", "w"))
imp = pd.Series(final.feature_importances_, BLEND).sort_values(ascending=False)
log("blender top features:\n" + imp.head(10).to_string())
del M, S, XS, D, A, s1, s1_text, ta, tb; gc.collect()
if DEV == "cuda": torch.cuda.empty_cache()


In [ ]:
# ---------- 1. v1 model on every test business, one batch at a time, saved to disk (safe to resume) ----------
PARTS = f"{WORK}/parts_v3"; os.makedirs(PARTS, exist_ok=True)
meta = json.load(open(f"{WORK}/model_v3_meta.json"))
boosters = [lgb.Booster(model_file=f"{WORK}/stage1_{k}.txt") for k in (0, 1)]
blender = lgb.Booster(model_file=f"{WORK}/blender.txt")
for src in ("test_source2", "test_source3"):            # clean both sources first (saved to disk)
    log("cleaning", src); B = load_src(src); log("cleaned", src, B.shape); del B; gc.collect()
log("loading test Source 1"); T = load(ROOT, "test_source1")
if SMOKE: T = T.groupby("country").head(2000).reset_index(drop=True)
t_text = np.array(rec_text(T.business_name, T.business_address), dtype=object)
A = prepare(T); log("prepared", A.shape); del T
print(A.country.value_counts().to_string())
s1_ids = A.entity_id.to_numpy()
for src, is_s3 in (("test_source2", 0), ("test_source3", 1)):
    todo = [(c, i) for c in A.country.unique() for i in range(0, int((A.country == c).sum()), BATCH)
            if not os.path.exists(f"{PARTS}/sel_{src}_{c}_{i}.parquet")]
    if not todo:
        log(src, "already done"); continue
    B = load_src(src).drop(columns=["rn", "ra"])
    if SMOKE: B = B[np.random.default_rng(1).random(len(B)) < 0.25].reset_index(drop=True)
    for country in A.country.unique():                 # any label works, France included
        a_rows = np.flatnonzero(A.country.to_numpy() == country)
        Bc = B[B.country == country].reset_index(drop=True)
        if len(Bc) == 0 or not any(c == country for c, _ in todo): continue
        idx = Index(Bc); bids = Bc.entity_id.to_numpy()
        for i in range(0, len(a_rows), BATCH):
            out = f"{PARTS}/sel_{src}_{country}_{i}.parquet"
            if os.path.exists(out): continue
            rows = a_rows[i:i + BATCH]; Ac = A.iloc[rows].reset_index(drop=True)
            P = idx.query(Ac)
            F = features(P, Ac, Bc, is_s3)[meta["v1_features"]] if len(P) else None
            p = p1_of(boosters, F) if len(P) else np.zeros(0, np.float32)
            s1r = rows[P.a.to_numpy()].astype(np.int32); cid = bids[P.b.to_numpy()]
            pd.DataFrame({"s1_row": s1r, "cand_id": cid, "p1": p}).to_parquet(f"{PARTS}/all_{src}_{country}_{i}.parquet")
            k = p >= CE_MINP                            # the rest can never be chosen: not kept
            sel = F[k].reset_index(drop=True) if len(P) else pd.DataFrame(columns=meta["v1_features"])
            sel["s1_row"], sel["cand_id"], sel["p1"], sel["src"] = s1r[k], cid[k], p[k], src
            sel.to_parquet(out)
            log(src, country, f"batch {i // BATCH + 1}/{-(-len(a_rows) // BATCH)}", "pairs", len(P), "kept", int(k.sum()))
            del P, F, p, sel; gc.collect()
        del idx, Bc; gc.collect()
    del B; gc.collect()

# ---------- 2. language model re-checks the best candidates of every business ----------
S = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(f"{PARTS}/sel_*.parquet"))], ignore_index=True)
S = S[select_for_ce(S, "s1_row")].reset_index(drop=True)
log("re-check set:", len(S), "pairs,", round(len(S) / len(A), 1), "per business")
tok, cem = ce_load(f"{WORK}/ce_model")
texts = {src: pl.read_parquet(f"{WORK}/clean_{src}.parquet", columns=["entity_id", "rn", "ra"])
         for src in ("test_source2", "test_source3")}
CH = 1_000_000; t0 = time.time()
for i in range(0, len(S), CH):
    out = f"{PARTS}/ce_{i}.npy"
    if os.path.exists(out): continue
    C = S.iloc[i:i + CH]; tb = np.empty(len(C), object)
    for src, tt in texts.items():
        m = (C.src == src).to_numpy()
        if m.any():
            q = C.cand_id.to_numpy()[m]
            t = pl.DataFrame({"entity_id": q, "_i": np.arange(len(q))}).join(tt, on="entity_id", how="left")
            t = t.sort("_i").to_pandas()
            tb[m] = rec_text(t.rn.fillna(""), t.ra.fillna(""))
    np.save(out, ce_score(tok, cem, t_text[C.s1_row.to_numpy()], tb))
    log(f"language model: {min(i + CH, len(S))}/{len(S)} pairs ({(time.time() - t0) / 60:.1f} min)")
del texts; gc.collect()
S["ce"] = np.concatenate([np.load(f"{PARTS}/ce_{i}.npy") for i in range(0, len(S), CH)])

# ---------- 3. final decision and output files ----------
S = add_ctx(S, "s1_row")
S["p"] = (blender.predict(S[meta["blend_features"]].to_numpy(np.float32), num_threads=N_CPU)
          if meta["use_blend"] else S.p1)
match = decide(S[["s1_row", "cand_id", "p"]].rename(columns={"s1_row": "s1_id"}), meta["threshold"])
os.makedirs(f"{WORK}/output", exist_ok=True)
with open(f"{WORK}/output/matching_results.tsv", "w", encoding="utf-8") as fm:
    fm.write("source1_entity_id\tmatched_entity_ids\n")
    for r, s in enumerate(s1_ids):
        fm.write(s + "\t" + ",".join(sorted(match.get(r, ()))) + "\n")
# candidate list, one batch at a time (all ~150M pairs at once would need too much memory)
cline = np.full(len(s1_ids), "", dtype=object)
batches = sorted({f.split("all_test_source")[1][2:] for f in glob.glob(f"{PARTS}/all_*.parquet")})
for bname in batches:
    files = [f for f in (f"{PARTS}/all_test_source2_{bname}", f"{PARTS}/all_test_source3_{bname}") if os.path.exists(f)]
    g = pl.concat([pl.read_parquet(x, columns=["s1_row", "cand_id"]) for x in files]) \
          .group_by("s1_row").agg(pl.col("cand_id").unique(maintain_order=True))
    for r, c in zip(g["s1_row"].to_list(), g["cand_id"].to_list()):
        cline[r] = ",".join(c)
    del g; gc.collect()
with open(f"{WORK}/output/candidate_pairs.tsv", "w", encoding="utf-8") as fc:
    fc.write("source1_entity_id\tcandidate_entity_ids\n")
    for s, c in zip(s1_ids, cline):
        fc.write(s + "\t" + c + "\n")
del cline; gc.collect()
n = sum(1 for r in range(len(s1_ids)) if match.get(r))
log("done:", len(s1_ids), "rows,", n, "with matches,", len(s1_ids) - n, "empty; avg matches",
    round(sum(len(v) for v in match.values()) / len(s1_ids), 2))


In [ ]:
# candidate_pairs check is skipped (none.tsv): it needs a lot of memory
validator = glob.glob(f'{ROOT}/**/validate_submission.py', recursive=True)[0]
test_dir = os.path.dirname(find_file(ROOT, 'test_source1')[0])
!python3 {validator} --matching {WORK}/output/matching_results.tsv --candidate none.tsv --test-dir {test_dir}

**Done.** Open this version's **Output** tab, go into `output/`, and download **`matching_results.tsv`**. Keep `candidate_pairs.tsv` for the final zip.